In [1]:
# @title 1. Install System Dependencies & Clone ComfyUI
import os
from IPython.display import clear_output

# 1. Install system tools
!apt-get update -y && apt-get install -y ffmpeg libgl1-mesa-glx aria2

# 2. Clone ComfyUI
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/Comfy-Org/ComfyUI.git
    print("✅ ComfyUI cloned.")
else:
    print("ℹ️ ComfyUI already exists.")

# 3. Clone ComfyUI Manager (Essential for installing missing nodes)
if not os.path.exists("ComfyUI/custom_nodes/ComfyUI-Manager"):
    %cd ComfyUI/custom_nodes
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager.git
    %cd ../..
    print("✅ ComfyUI Manager installed.")

# 4. Install Python Requirements
%cd ComfyUI
!pip install -r requirements.txt
!pip install pyngrok  # Needed for the tunnel later

print("✅ Installation complete.")
clear_output()

In [2]:
# @title 2. Apply Storage Hack (Move Models to /tmp)
import os
import shutil

# Define paths
BASE_DIR = "/kaggle/working/ComfyUI"
TEMP_DIR = "/tmp/ComfyUI_Storage" # /tmp usually has ~Disk space on Kaggle

# Create temp directory
os.makedirs(TEMP_DIR, exist_ok=True)

# Function to move folder to temp and symlink back
def setup_storage_link(folder_name):
    src = os.path.join(BASE_DIR, folder_name)
    dst = os.path.join(TEMP_DIR, folder_name)
    
    # If the link already exists, skip
    if os.path.islink(src):
        print(f"ℹ️ {folder_name} is already linked.")
        return

    # If the source folder exists (real folder), move it to temp
    if os.path.exists(src):
        # Create destination if missing
        if not os.path.exists(dst):
            print(f"📦 Moving existing {folder_name} to {dst}...")
            shutil.move(src, dst)
        else:
            # If both exist, merge/delete source (safe for empty default folders)
            print(f"⚠️ {dst} exists. Merging/Replacing {src}...")
            shutil.rmtree(src)
    
    # Create the symlink
    if not os.path.exists(src):
        os.symlink(dst, src)
        print(f"🔗 Linked {src} -> {dst}")

# Apply to heavy folders
setup_storage_link("models")
setup_storage_link("input")
setup_storage_link("output")

print("✅ Storage hack applied successfully.")

📦 Moving existing models to /tmp/ComfyUI_Storage/models...
🔗 Linked /kaggle/working/ComfyUI/models -> /tmp/ComfyUI_Storage/models
📦 Moving existing input to /tmp/ComfyUI_Storage/input...
🔗 Linked /kaggle/working/ComfyUI/input -> /tmp/ComfyUI_Storage/input
📦 Moving existing output to /tmp/ComfyUI_Storage/output...
🔗 Linked /kaggle/working/ComfyUI/output -> /tmp/ComfyUI_Storage/output
✅ Storage hack applied successfully.


In [4]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [5]:
!apt-get update -y
!apt-get install -y aria2 ffmpeg libgl1
!which aria2c

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease                         
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease     
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease                 
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease                     
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease               
Hit:8 http://security.ubuntu.com/ubuntu noble-security InRelease               
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease   
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide

In [7]:
import os, subprocess, shutil

BASE = "/kaggle/working/ComfyUI/models"
DIRS = {
    "diffusion_models": f"{BASE}/diffusion_models",
    "vae": f"{BASE}/vae",
    "text_encoders": f"{BASE}/text_encoders",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

# disk check (models dir symlinks to /tmp)
free_gb = shutil.disk_usage("/tmp").free / 1e9
print(f"Free in /tmp: {free_gb:.1f} GB (need ~18+ GB)")

REPO = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"

downloads = [
    # Wan 2.2 5B text+image to video, fp16 (T4 has no bf16)
    (f"{REPO}/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
     f"{DIRS['diffusion_models']}/wan2.2_ti2v_5B_fp16.safetensors"),

    # Wan 2.2 VAE
    (f"{REPO}/vae/wan2.2_vae.safetensors",
     f"{DIRS['vae']}/wan2.2_vae.safetensors"),

    # UMT5 text encoder
    (f"{REPO}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     f"{DIRS['text_encoders']}/umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
]

def dl(url, out_path):
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1024 * 1024:
        print("✅ Exists:", os.path.basename(out_path))
        return
    hdr = f'--header="Authorization: Bearer {os.environ["HF_TOKEN"]}"' if os.environ.get("HF_TOKEN") else ""
    cmd = f'aria2c -x 16 -s 16 -c {hdr} -o "{os.path.basename(out_path)}" -d "{os.path.dirname(out_path)}" "{url}"'
    print("⬇️", os.path.basename(out_path))
    subprocess.run(cmd, shell=True, check=True)

for url, outp in downloads:
    dl(url, outp)

!ls -lh "{DIRS['diffusion_models']}" "{DIRS['vae']}" "{DIRS['text_encoders']}"

Free in /tmp: 1036.2 GB (need ~18+ GB)
⬇️ wan2.2_ti2v_5B_fp16.safetensors

11/14 05:20:04 [NOTICE] Downloading 1 item(s)

11/14 05:20:04 [NOTICE] CUID#7 - Redirecting to https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors

11/14 05:20:04 [NOTICE] CUID#7 - Redirecting to https://us.gcp.cdn.hf.co/xet-bridge-us/6885cd8c6963bab90aab7f6f/2d6bf9d352156e0aa55626945366a75679964d589aaa7555a23bdaf173925871?user_id=67a358a863acd308868268fa&xip=MG4XYSwPc-Y&X-Xet-Cas-Uid=67a358a863acd308868268fa&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27wan2.2_ti2v_5B_fp16.safetensors%3B+filename%3D%22wan2.2_ti2v_5B_fp16.safetensors%22%3B&Expires=1794637204&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjg4NWNkOGM2OTYzYmFiOTBhYWI3ZjZmLzJkNmJmOWQzNTIxNTZlMGFhNTU2MjY5NDUzNjZhNzU2Nzk5NjRkNTg5YWFhNzU1NWEyM2JkYWYxNzM5MjU4NzFcXD91c2VyX2lkPTY3YTM1OGE4NjNhY2QzMDg4NjgyNjhmYSZ4aXA9TUc0WFlTd1BjLV

In [ ]:
import subprocess, re, time, os, urllib.request

PORT = 8188
COMFY_DIR = "/kaggle/working/ComfyUI"
COMFY_LOG = "/kaggle/working/comfy.log"
PINGGY_LOG = "/kaggle/working/pinggy.log"

os.system("pkill -f 'python main.py'; pkill -f 'a.pinggy.io'")
time.sleep(1)

comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", str(PORT)],
    cwd=COMFY_DIR, stdout=open(COMFY_LOG, "w"), stderr=subprocess.STDOUT,
)

# wait for ComfyUI up
for _ in range(90):
    if comfy.poll() is not None:
        print(open(COMFY_LOG).read()[-3000:])
        raise RuntimeError("ComfyUI died, see log above")
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}", timeout=2)
        print("ComfyUI up")
        break
    except Exception:
        time.sleep(2)
else:
    print(open(COMFY_LOG).read()[-3000:])
    raise RuntimeError("ComfyUI never came up")

pinggy = subprocess.Popen(
    ["ssh", "-T", "-p", "443", "-R0:127.0.0.1:%d" % PORT,
     "-o", "StrictHostKeyChecking=no",
     "-o", "ServerAliveInterval=30",
     "-o", "ServerAliveCountMax=3",
     "a.pinggy.io"],
    stdin=subprocess.DEVNULL,
    stdout=open(PINGGY_LOG, "w"), stderr=subprocess.STDOUT,
)

url = None
for _ in range(60):
    txt = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", open(PINGGY_LOG).read())
    m = re.search(r"https://[^\s]*pinggy\.(?:link|live|io)[^\s]*", txt)
    if m and "dashboard" not in m.group(0):
        url = m.group(0); break
    time.sleep(1)

print("URL:", url) if url else print("No URL. Pinggy log:\n", txt)

while True:   # keep cell alive, tunnel dies if kernel idle-stops
    time.sleep(60)

ComfyUI up
No URL. Pinggy log:
Allocated port 2 for remote forward to 127.0.0.1:8188
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
https://yyuza-104-155-188-139.free.pinggy.net
https://zfrzy-104-155-188-139.run.pinggy-free.link

